# 🌍 GeoCongo AI - Pipeline de Test Colab
Ce notebook permet de tester le moteur d'analyse géologique de GeoCongo AI. 

**Fonctionnalités incluses :**
1. Connexion à Google Earth Engine (GEE) pour l'imagerie Sentinel-2.
2. Extraction de données multispectrales sur la RDC.
3. Simulation/Exécution d'inférence (Prithvi/SAM 2).
4. Vectorisation Raster vers GeoJSON.
5. Visualisation interactive sur carte.

In [1]:
# 1. Installation des dépendances
!pip install rasterio transformers torch ultralytics shapely folium requests

  Using cached folium-0.20.0-py2.py3-none-any.whl.metadata (4.2 kB)
  Using cached branca-0.8.2-py3-none-any.whl.metadata (1.7 kB)
  Using cached xyzservices-2026.3.0-py3-none-any.whl.metadata (4.1 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 422.0 kB/s  0:00:03 eta 0:00:01
Using cached folium-0.20.0-py2.py3-none-any.whl (113 kB)
Using cached branca-0.8.2-py3-none-any.whl (26 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 833.4/833.4 kB 627.1 kB/s  0:00:01eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 MB 1.4 MB/s  0:00:51m0:00:0100:020m
Using cached xyzservices-2026.3.0-py3-none-any.whl (94 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [ultralytics] [ultralytics]me-32]


In [2]:
# 2. Authentification Google Earth Engine
import ee
import os
import json
import rasterio
import numpy as np
import requests
import zipfile
import io
import tempfile
import shutil
from rasterio.features import shapes
from shapely.geometry import shape
import folium

try:
    ee.Authenticate()
    ee.Initialize()
    print("✅ Earth Engine initialisé avec succès.")
except Exception as e:
    print(f"❌ Erreur d'initialisation GEE : {e}")

/home/gerard/Documents/GeoKivuDoc/geocongoai-api/venv/lib/python3.10/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.13) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)



Successfully saved authorization token.
✅ Earth Engine initialisé avec succès.


## 🛠️ Implémentation des Services (Version Légère)

In [3]:
class AIServiceColab:
    """Version adaptée pour Colab sans infrastructure Vertex AI Batch."""
    def __init__(self):
        self.temp_dir = tempfile.mkdtemp()
        
    async def fetch_satellite_data(self, bbox, analysis_type='minéraux'):
        print(f"🛰️ Récupération des données Sentinel-2 pour {bbox} via GEE...")
        region = ee.Geometry.BBox(bbox[0], bbox[1], bbox[2], bbox[3])
        
        # Bandes Prithvi: Blue, Green, Red, NIR, SWIR1, SWIR2
        bands = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12']
        
        collection = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
                      .filterBounds(region)
                      .filterDate('2023-01-01', '2023-12-31')
                      .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 10))
                      .sort("CLOUDY_PIXEL_PERCENTAGE"))

        image = collection.first().select(bands)
        
        url = image.getDownloadURL({
            'scale': 10,
            'crs': 'EPSG:4326',
            'region': region,
            'format': 'GEO_TIFF'
        })
        
        response = requests.get(url)
        download_path = os.path.join(self.temp_dir, "gee_data.tif")
        
        with zipfile.ZipFile(io.BytesIO(response.content)) as z:
            extracted = z.namelist()[0]
            z.extract(extracted, self.temp_dir)
            shutil.move(os.path.join(self.temp_dir, extracted), download_path)
            
        return download_path

    def run_mock_inference(self, raster_path):
        """Simule une sortie d'IA (classification) basée sur le seuillage NIR/Red (NDVI inverse)."""
        output_path = raster_path.replace(".tif", "_inference.tif")
        with rasterio.open(raster_path) as src:
            profile = src.profile
            # Lecture simplifiée pour test
            data = src.read()
            # Simulation d'un masque de détection (ex: zones à forte réflectance SWIR)
            mock_result = (data[4] > 3000).astype(np.uint8) 
            
            profile.update(count=1, dtype='uint8')
            with rasterio.open(output_path, 'w', **profile) as dst:
                dst.write(mock_result, 1)
        return output_path

In [4]:
class GeoServiceColab:
    """Utilise Rasterio pour la vectorisation (alternative à PyQGIS)."""
    def vectorize(self, raster_path):
        print(f"📐 Vectorisation du résultat...")
        with rasterio.open(raster_path) as src:
            image = src.read(1)
            results = (
                {'properties': {'class': int(v)}, 'geometry': s}
                for i, (s, v) in enumerate(shapes(image, mask=(image > 0), transform=src.transform))
            )
            
        features = []
        for res in results:
            features.append({
                "type": "Feature",
                "properties": res['properties'],
                "geometry": res['geometry']
            })
        return {"type": "FeatureCollection", "features": features}

## 🚀 Exécution du Test
Nous allons analyser une zone près de **Bukavu (Est de la RDC)**.

In [5]:
import asyncio

# Configuration de la zone (minx, miny, maxx, maxy)
BBOX_BUKAVU = [28.80, -2.55, 28.90, -2.45]

async def run_pipeline():
    ai = AIServiceColab()
    geo = GeoServiceColab()
    
    # 1. Fetch
    raster_path = await ai.fetch_satellite_data(BBOX_BUKAVU)
    
    # 2. Inférence (Mock pour le test Colab rapide)
    inf_path = ai.run_mock_inference(raster_path)
    
    # 3. Vectorisation
    geojson_results = geo.vectorize(inf_path)
    
    print(f"✅ Analyse terminée. {len(geojson_results['features'])} entités détectées.")
    return geojson_results, BBOX_BUKAVU

# Lancement de l'analyse
loop = asyncio.get_event_loop()
results, area = loop.run_until_complete(run_pipeline())

RuntimeError: This event loop is already running

## 🗺️ Visualisation des Résultats

In [ ]:
m = folium.Map(location=[(area[1]+area[3])/2, (area[0]+area[2])/2], zoom_start=13)

# Zone d'étude
folium.Rectangle(
    bounds=[[area[1], area[0]], [area[3], area[2]]], 
    color="blue", 
    fill=False, 
    dash_array='5, 5'
).add_to(m)

# Résultats de l'IA (en rouge)
folium.GeoJson(
    results, 
    style_function=lambda x: {'color': 'red', 'fillColor': 'red', 'weight': 1, 'fillOpacity': 0.5}
).add_to(m)

m